# Step Detection — LOWESS Smoother & Multi-Run Comparison

This notebook implements step detection for all available measurement runs (R1, R2, R3)  
using a **LOWESS smoother** (Locally Weighted Scatterplot Smoothing) as the preprocessing filter.  

LOWESS is conceptually grounded in the least squares curve fitting framework from the lecture  
*(leastsquareOptimization)*: at each time step, a locally weighted residual sum Σ wᵢ·rᵢ² is minimised  
over a local neighbourhood — no scipy required.

**Pipeline:**
1. Load accelerometer data from DB (all runs)
2. Compute Euclidean norm
3. Apply LOWESS smoothing
4. Stationarity check (ADF + KPSS)
5. ACF / PACF analysis
6. Peak detection (numpy + find_peaks via numpy)
7. Multi-run comparison
8. Write results to DB (`steps` table)

In [ ]:
# ── Imports ───────────────────────────────────────────────────
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.nonparametric.smoothers_lowess import lowess

print('Imports OK ✓')

In [ ]:
# ── Config ────────────────────────────────────────────────────
DB_PATH  = '../data/emi_nav.db'
RUN_IDS  = ['R1', 'R2', 'R3']
LOWESS_FRAC = 0.05   # 5% of samples per local regression window
THRESHOLD_K = 0.5    # dynamic threshold: mean + K * std
MIN_STEP_MS = 300    # physiological minimum between steps [ms]

## Step 1 — Load accelerometer data from DB

In [ ]:
# ── Load all runs ─────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

runs = {}
for run_id in RUN_IDS:
    df = pd.read_sql_query(f"""
        SELECT timestamp_ms, x, y, z
        FROM imu
        WHERE run_id = '{run_id}' AND sensor = 'accel'
        ORDER BY timestamp_ms
    """, conn)
    runs[run_id] = df
    print(f'{run_id}: {len(df):,} accelerometer samples loaded')

conn.close()

## Step 2 — Euclidean Norm & Sampling Rate

In [ ]:
# ── Norm + sampling rate for each run ─────────────────────────
fs_per_run = {}

for run_id, df in runs.items():
    df['norm']   = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
    df['time_s'] = (df['timestamp_ms'] - df['timestamp_ms'].min()) / 1000
    dt_ms = df['timestamp_ms'].diff().median()
    fs    = 1000 / dt_ms
    fs_per_run[run_id] = fs
    print(f'{run_id}: fs = {fs:.1f} Hz  |  duration = {df["time_s"].max():.1f} s')

**Interpretation**

The Euclidean norm ||a|| = sqrt(x² + y² + z²) collapses the three-axis signal
into one rotation-invariant scalar. At rest the norm sits near 9.81 m/s² (gravity).
Walking produces periodic peaks above that baseline — these are the step impulses
to be detected. Estimating the sampling rate per run is essential because the LOWESS
window fraction and the minimum peak distance are derived from fs.

## Step 3 — LOWESS Smoothing

In [ ]:
# ── LOWESS filter (statsmodels — no scipy) ────────────────────
for run_id, df in runs.items():
    smoothed = lowess(
        df['norm'].values,
        df['time_s'].values,
        frac=LOWESS_FRAC,
        return_sorted=False
    )
    df['norm_lowess'] = smoothed

# ── Plot all runs ─────────────────────────────────────────────
fig, axes = plt.subplots(len(RUN_IDS), 1, figsize=(13, 4 * len(RUN_IDS)), sharex=False)

for ax, run_id in zip(axes, RUN_IDS):
    df = runs[run_id]
    ax.plot(df['time_s'], df['norm'],        alpha=0.25, label='raw',    color='gray')
    ax.plot(df['time_s'], df['norm_lowess'], label=f'LOWESS (frac={LOWESS_FRAC})',
            linewidth=2, color='steelblue')
    ax.set_title(f'[{run_id}] Accelerometer Magnitude — LOWESS Smoothing')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||a|| [m/s²]')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

**Interpretation**

LOWESS (Locally Weighted Scatterplot Smoothing) applies a local linear regression
at each point, weighting nearby samples more strongly than distant ones.
This is conceptually equivalent to the local least squares fitting framework
from the lecture *(leastsquareOptimization)*: at each time step t,
a weighted residual sum Σ wᵢ·rᵢ² is minimised over a local neighbourhood.

The parameter `frac = 0.05` corresponds to approximately 100 ms at 50 Hz,
which preserves step-induced peaks at 1–2 Hz while suppressing high-frequency noise.
Unlike a simple moving average, LOWESS adapts to local signal density and
is robust against isolated outlier samples caused by sensor spikes.

## Step 4 — Stationarity Check (ADF + KPSS)

In [ ]:
# ── Stationarity per run ──────────────────────────────────────
stationarity_results = []

for run_id, df in runs.items():
    sig = df['norm_lowess'].dropna()

    adf_stat, adf_p, *_ = adfuller(sig)
    kpss_stat, kpss_p, *_ = kpss(sig, regression='c', nlags='auto')

    stationary = (adf_p < 0.05) and (kpss_p > 0.05)

    if not stationary:
        df['signal_det'] = df['norm_lowess'].diff()
        decision = '1st-order differencing applied'
    else:
        df['signal_det'] = df['norm_lowess']
        decision = 'signal used directly'

    stationarity_results.append({
        'Run':      run_id,
        'ADF p':    round(adf_p,  4),
        'KPSS p':   round(kpss_p, 4),
        'Stationary': stationary,
        'Decision': decision
    })
    print(f'{run_id} | ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f} | → {decision}')

pd.DataFrame(stationarity_results)

**Interpretation**

ADF tests H₀: non-stationary — rejection (p < 0.05) indicates stationarity.
KPSS tests H₀: stationary — non-rejection (p > 0.05) confirms stationarity.
Both tests are applied following the analysis workflow from the lecture
*(01_TimeSeries, Section 6.2)*: stationarity must be verified before
inspecting ACF/PACF plots. If either test indicates non-stationarity,
first-order differencing Δxₜ = xₜ − xₜ₋₁ is applied exactly once
to remove drift without amplifying noise through over-differencing.

## Step 5 — ACF / PACF Analysis

In [ ]:
# ── ACF / PACF per run ────────────────────────────────────────
for run_id, df in runs.items():
    sig = df['signal_det'].dropna()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))
    plot_acf( sig, lags=60, ax=ax1,
              title=f'[{run_id}] ACF  – LOWESS-Smoothed Accelerometer Magnitude')
    plot_pacf(sig, lags=60, ax=ax2,
              title=f'[{run_id}] PACF – LOWESS-Smoothed Accelerometer Magnitude')
    ax1.set_xlabel('Lag [samples]'); ax1.set_ylabel('Autocorrelation')
    ax2.set_xlabel('Lag [samples]'); ax2.set_ylabel('Partial Autocorrelation')
    plt.tight_layout()
    plt.show()

**Interpretation**

Accelerometer step signals typically exhibit MA-type behaviour:
a sharp ACF cut-off after 1–2 lags with gradual PACF decay.
This reflects the physical reality that a single footstep is a short-lived impulse —
the signal returns to baseline quickly after each peak.
Seasonal spikes in the ACF at regular lag multiples indicate the stride frequency
and can be used to cross-validate the detected step count.
Consistent patterns across R1, R2, R3 indicate reproducible signal behaviour.

## Step 6 — Peak Detection (numpy only)

In [ ]:
# ── Peak detection — pure numpy, no scipy ────────────────────
def find_peaks_numpy(signal, height, min_distance):
    """
    Detect local maxima above `height` with minimum separation `min_distance`.
    Pure numpy implementation — no scipy dependency.
    """
    peaks = []
    n = len(signal)
    for i in range(1, n - 1):
        if signal[i] > signal[i - 1] and signal[i] > signal[i + 1]:
            if signal[i] >= height:
                if not peaks or (i - peaks[-1]) >= min_distance:
                    peaks.append(i)
                elif signal[i] > signal[peaks[-1]]:
                    peaks[-1] = i   # replace with higher peak in same window
    return np.array(peaks)


step_results = {}

for run_id, df in runs.items():
    fs       = fs_per_run[run_id]
    sig_vals = df['signal_det'].dropna().values
    sig_time = df['time_s'].dropna().values

    threshold = sig_vals.mean() + THRESHOLD_K * sig_vals.std()
    min_dist  = int((MIN_STEP_MS / 1000) * fs)

    peaks     = find_peaks_numpy(sig_vals, height=threshold, min_distance=min_dist)
    step_times = sig_time[peaks]

    step_results[run_id] = {
        'peaks':       peaks,
        'step_times':  step_times,
        'sig_vals':    sig_vals,
        'sig_time':    sig_time,
        'threshold':   threshold,
        'n_steps':     len(peaks),
        'freq':        len(peaks) / df['time_s'].max()
    }

    print(f'{run_id}: {len(peaks)} steps detected  |  '
          f'freq = {len(peaks)/df["time_s"].max():.2f} steps/s  |  '
          f'threshold = {threshold:.3f}')

In [ ]:
# ── Plot per run ──────────────────────────────────────────────
fig, axes = plt.subplots(len(RUN_IDS), 1, figsize=(13, 4 * len(RUN_IDS)), sharex=False)

for ax, run_id in zip(axes, RUN_IDS):
    r = step_results[run_id]
    ax.plot(r['sig_time'], r['sig_vals'], alpha=0.8, label='signal')
    ax.plot(r['step_times'], r['sig_vals'][r['peaks']],
            'x', color='red', markersize=9, label=f'steps (n={r["n_steps"]})')
    ax.axhline(r['threshold'], color='orange', linestyle='--',
               label=f'threshold (μ + {THRESHOLD_K}σ)')
    ax.set_title(f'[{run_id}] Step Detection — LOWESS-Smoothed Signal')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||a|| [m/s²]')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

**Interpretation**

Each red cross marks one detected step. The dynamic threshold (μ + 0.5σ) adapts
to the signal level across different runs and avoids hard-coded constants that would
fail when walking speed or device placement varies. The 300 ms minimum distance
between peaks reflects the physiological lower bound of stride duration (cadence ≤ 3.3 steps/s).

**Tuning guide:**
- `LOWESS_FRAC` too large → peaks over-smoothed → missed steps → decrease
- `LOWESS_FRAC` too small → noise not suppressed → false positives → increase
- `THRESHOLD_K` too high → missed steps → decrease (e.g. 0.3)
- `MIN_STEP_MS` too small → double-counted steps → increase (e.g. 400 ms)

## Step 7 — Multi-Run Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────
summary = pd.DataFrame([
    {
        'Run':            run_id,
        'Duration [s]':   round(runs[run_id]['time_s'].max(), 1),
        'Steps detected': step_results[run_id]['n_steps'],
        'Step freq [/s]': round(step_results[run_id]['freq'], 3),
        'Threshold':      round(step_results[run_id]['threshold'], 3),
        'fs [Hz]':        round(fs_per_run[run_id], 1)
    }
    for run_id in RUN_IDS
])
print(summary.to_string(index=False))
summary

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

runs_list  = list(step_results.keys())
n_steps    = [step_results[r]['n_steps'] for r in runs_list]
step_freqs = [step_results[r]['freq']    for r in runs_list]

ax1.bar(runs_list, n_steps, color='steelblue', edgecolor='black')
ax1.set_title('Total Steps per Run')
ax1.set_xlabel('Run')
ax1.set_ylabel('Steps detected')
ax1.grid(axis='y')
for i, v in enumerate(n_steps):
    ax1.text(i, v + 1, str(v), ha='center', fontweight='bold')

ax2.bar(runs_list, step_freqs, color='coral', edgecolor='black')
ax2.set_title('Step Frequency per Run')
ax2.set_xlabel('Run')
ax2.set_ylabel('Steps / second')
ax2.grid(axis='y')
for i, v in enumerate(step_freqs):
    ax2.text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretation**

Comparing step counts and frequencies across all three runs reveals whether the
detection pipeline is consistent across different starting positions, paths, and
walking speeds — as required by the assignment. Runs with floor transitions are
expected to show brief gaps in step detection during stair climbing due to the
different acceleration pattern compared to flat-floor walking.
Step frequency should fall in the physiologically plausible range of 1.2–2.2 steps/s.

## Step 8 — Write Results to DB (`steps` table)

In [ ]:
# ── Create steps table ────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS steps;
CREATE TABLE steps (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id          TEXT    NOT NULL,
    timestamp_ms    INTEGER NOT NULL,
    accel_norm      REAL,
    step_index      INTEGER,
    FOREIGN KEY (run_id) REFERENCES runs(run_id)
);
""")
conn.commit()
print("Table 'steps' created ✓")

In [ ]:
# ── Insert all runs ───────────────────────────────────────────
total_written = 0

for run_id, df in runs.items():
    r    = step_results[run_id]
    idx  = r['peaks']

    # Map peak indices back to original df timestamps
    sig_subset = df['signal_det'].dropna().reset_index(drop=True)
    norm_vals  = runs[run_id]['norm_lowess'].dropna().reset_index(drop=True)
    ts_vals    = runs[run_id]['timestamp_ms'].reset_index(drop=True)

    rows = [
        (
            run_id,
            int(ts_vals.iloc[i]),
            float(norm_vals.iloc[i]),
            int(step_num)
        )
        for step_num, i in enumerate(idx)
    ]

    cur.executemany("""
        INSERT INTO steps (run_id, timestamp_ms, accel_norm, step_index)
        VALUES (?, ?, ?, ?)
    """, rows)
    conn.commit()

    total_written += len(rows)
    print(f'{run_id}: {len(rows)} steps written to DB ✓')

conn.close()
print(f'\nTotal: {total_written} step records written to steps table ✓')

In [ ]:
# ── Verify DB write ───────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
verification = pd.read_sql_query("""
    SELECT run_id, COUNT(*) as step_count
    FROM steps
    GROUP BY run_id
    ORDER BY run_id
""", conn)
conn.close()

print('DB verification:')
print(verification.to_string(index=False))
verification

**Interpretation**

The `steps` table stores one record per detected step, linked to its run via `run_id`
and to the raw IMU data via `timestamp_ms`. This structure allows the particle filter
in the next pipeline stage to query step events by run and timestamp, and use
`accel_norm` as an optional proxy for step length estimation (higher norm ≈ stronger
push-off ≈ longer step). The DB schema is consistent with the existing `imu` and
`ble_rssi` tables — all linked via `run_id` and synchronised via `timestamp_ms`.